# 03. Prototype the AI Career Mentor RAG Chain
**SmartHire GenAI Capstone Project**

This notebook demonstrates:
1. Loading markdown career notes and guides
2. Chunking text with boundary preservation and overlaps
3. Embedding chunks and indexing them with FAISS
4. Running semantic retrieval against candidate queries
5. Formatting context for LLM generation and verifying ground truth sources

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

notes_dir = Path('../data/career_notes')
doc_chunks = []

for p in notes_dir.glob('*.md'):
    text = p.read_text(encoding='utf-8')
    # Simple paragraph chunker
    paras = [para.strip() for para in text.split('\n\n') if len(para.strip()) > 60]
    for i, para in enumerate(paras):
        doc_chunks.append({'source': p.name, 'chunk_id': i, 'text': para})

print(f"Extracted {len(doc_chunks)} chunks from {len(list(notes_dir.glob('*.md')))} career guides.")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c['text'] for c in doc_chunks]
chunk_vectors = model.encode(texts, normalize_embeddings=True)

index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors.astype(np.float32))
print("Career notes vectorstore populated.")

In [ ]:
# Test RAG Retrieval
query = "How do I write Google X-Y-Z bullet points on my resume?"
query_vec = model.encode([query], normalize_embeddings=True)

scores, indices = index.search(query_vec.astype(np.float32), 3)

print(f"Query: {query}\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    chunk = doc_chunks[idx]
    print(f"Rank #{rank} | Source: {chunk['source']} | Relevancy: {score:.3f}")
    print(f"{chunk['text'][:200]}...\n")